[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/spinning_top/spinning_top.ipynb)

# A Spinning Top

This example spins a top on flat ground. The top is a heavy disc on a stem with a rounded tip; spun upright with a slight tilt, it does not fall over but swings its axis around the vertical, precessing, while the air slowly takes its spin. Where its tip touches the ground it is pushed out of the ground and held against sliding by friction.

Every shape is a quadric, a map from points to planes: the top's three parts are solid ellipsoids, and the ground is the solid below a paraboloid. A part's lowest point against the ground is the pole of its tangent plane facing the ground, and how far that point lies below the ground is its depth. The top's inertia is built from a few mass points per part, each joined with its own motion under a rate, `points & points.commutator(Rate)`. The top moves by Lie-group steps of its motor, and after each step every contact is corrected along one force line: pressed out along the ground's normal and held against sliding within the friction cone, each correction the force line's own compliance, `twist & body`, with the top's inverse inertia.

In the notation of computational contact mechanics the corrections read as a position-based projection with a Coulomb friction cone, and the shapes as quadric surfaces $\mathbf p^{\mathsf T} Q\,\mathbf p \le 0$ in homogeneous coordinates.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from examples.animation import save_figure, show_animation
from examples.mechanics.spinning_top import render
from examples.wgsl import image_figure

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
# Directions are points at infinity: antivectors with no part along w.
Direction = Point.derive.degenerate
# A rate of motion, and a force line, the load along it.
Rate = ga.gatype.bivector()
Forque = ga.gatype.antibivector()
Inertia = ga.gatype((Forque, Rate))                                            # Forque <- Rate
InverseInertia = ga.gatype((Rate, Forque))                                     # Rate <- Forque
Motor = ga.gatype.rotor()
# A shape: a map from points to planes, whose pairing with a point is negative inside.
Quadric = ga.gatype((Plane, Point))
w = mv.w
axes = mv(Plane.derive.nondegenerate, np.eye(3))                               # [3] Plane
directions = mv(Direction, np.eye(3))                                          # [3] Direction
origin = w.dual()                                                              # [] Point
up = directions[2]                                                             # [] Direction
# The view: elevation and azimuth in degrees, the point it looks at, its half width, and its pixels across a top.
view, view_centre, view_extent, n_pixels = (30, -60), np.array([0.0, 0.0, 0.15]), 0.55, 260

## 1. The tops and the ground

Four tops are spun side by side: a base top, and three that each change one thing about it, a sharper tip, a heavier disc and slipperier ground. Each top's parts are a disc, a stem and a tip, each a solid ellipsoid about its centre, and the ground is the solid below a paraboloid, flat here, its curvature zero. Six mass points per part, placed along its axes, carry the part's mass, centroid and second moments. Each mass point joined with its own motion under a rate, `points & points.commutator(Rate)`, is its momentum, and the sum over the points is the top's inertia, a map from rates to forques, with no centre of mass or parallel-axis shift to place.

In [ ]:
# The disc and the stem, as in every top: their semi-axes; the tip's height; the parts' heights above the tip, disc,
# stem and tip; and the stem's and the tip's masses.
disc_semi_axes = np.array([0.25, 0.25, 0.04])
stem_semi_axes = np.array([0.02, 0.02, 0.15])
tip_height = 0.14
part_heights = np.array([0.10, 0.26, tip_height])
stem_mass, tip_mass = 0.05, 0.15
# Each top's tip radius and disc mass: the base top, a sharper tip, a heavier disc, and the base top again for the
# slipperier ground.
tip_radii = np.array([0.05, 0.02, 0.05, 0.05])
disc_masses = np.array([1.0, 1.0, 2.0, 1.0])
ground_curvature = 0.0

In [ ]:
def ellipsoid(centres: Point, semi_axes: np.ndarray) -> Quadric:
    """Solid ellipsoids about centres with the given semi-axes `[..., 3]`: the dyads of the planes through each centre
    across the axes, each over its semi-axis squared, less the plane at infinity's."""
    through = axes - w * (axes & centres[..., None])                           # [..., 3] Plane
    return (through * (through & Point) / semi_axes**2).sum(axis=-1) - w * (w & Point)


def bowl(curvature: float) -> Quadric:
    """The ground: solid below the paraboloid whose height is the curvature times the squared distance from the z
    axis; a bowl for positive curvature, a dome for negative, flat for none."""
    return 0.5 * (mv.z * (w & Point) + w * (mv.z & Point)) - curvature * (axes[:2] * (axes[:2] & Point)).sum()


def top(part_heights: np.ndarray, semi_axes: np.ndarray, part_masses: np.ndarray) -> tuple[Quadric, Inertia, np.ndarray]:
    """The parts of a top about its centre of mass `[..., parts]`, its inertia, and the centre of mass's height above
    the tip `[...]`, from the parts' heights above the tip and masses `[..., parts]`, and semi-axes `[..., parts, 3]`."""
    height = (part_heights * part_masses).sum(axis=-1) / part_masses.sum(axis=-1)
    centres = origin + up * (part_heights - height[..., None])                 # [..., parts] Point
    # Six mass points a part, each a sixth of its mass, out from its centre along its axes by the semi-axes times the
    # root of three fifths: together they carry the solid ellipsoid's mass, centroid and second moments.
    offsets = directions * (semi_axes * np.sqrt(3 / 5))                       # [..., parts, 3] Direction
    points = concatenate([centres[..., None] + offsets, centres[..., None] - offsets], axis=-1)   # [..., parts, 6] Point
    # Each mass point joined with its own motion under a rate is its momentum; summed over the points, the inertia.
    momenta = points & points.commutator(Rate) * (part_masses[..., None] / 6)  # [..., parts, 6] Forque <- Rate
    return ellipsoid(centres, semi_axes), momenta.sum(axis=-1).sum(axis=-1), height


# Every top's disc, stem and tip, the tip as round as its radius, and their masses.
semi_axes = np.stack(np.broadcast_arrays(disc_semi_axes, stem_semi_axes,
                                         np.stack(np.broadcast_arrays(tip_radii, tip_radii, tip_height), axis=-1)), axis=-2)   # [tops, parts, 3]
part_masses = np.stack(np.broadcast_arrays(disc_masses, stem_mass, tip_mass), axis=-1)   # [tops, parts]
parts, inertia, height = top(part_heights, semi_axes, part_masses)              # [tops, parts] Quadric, [tops] Inertia, [tops]
ground = bowl(ground_curvature)                                                 # [] Quadric

## 2. The contact point

The ground's form grows towards the air, and a point's polar plane under the ground is the gradient of that form there: read as a direction it is the ground's normal, at any point, on the ground or off it. A part's lowest point against that normal is the pole of its tangent plane on that side, and its depth below the ground is the ground's form there over the gradient's length. Here the tops stand in their starting pose, each lifted until its tip just clears the ground, then tilted.

In [ ]:
# The tilt the tops are spun at, and how far each tip starts clear of the ground.
tilt, clearance = 0.15, 0.002

In [ ]:
def ground_normal(ground: Quadric, points: Point) -> Direction:
    """The ground's unit normal at points, toward the air: a point's polar plane under the ground is the gradient of
    the ground's form there, read as a direction and divided by its length."""
    return ground(points).dual().cast(Direction) / ground(points).norm()


def lowest_point(surfaces: Quadric, normal: Direction) -> Point:
    """Each quadric's point furthest against the normal: the pole of its tangent plane on that side."""
    # Planes to their poles; the pole of the plane at infinity is the centre.
    dual = surfaces.inverse()
    centre = dual(w) / (w & dual(w))
    # The plane through the centre square to the normal has its pole at infinity, the direction conjugate to it,
    # along which the centre reaches the tangent planes parallel to it; scaled to reach the surface.
    plane = normal.dual() - w * (normal.dual() & centre)
    conjugate = dual(plane)
    return centre + conjugate / (-(plane & conjugate) * (w & dual(w))).square_root()


def contacts(motor: Motor, parts: Quadric, ground: Quadric) -> tuple[Point, Direction, Scalar]:
    """Each placed part's contact with the ground `[..., parts]`: its lowest point against the ground's normal at its
    centre, the ground's normal there, and how deep the point lies below the ground, negative in the air."""
    placed = motor[..., None] >> parts(motor[..., None] << Point)             # [..., parts] Quadric
    centres = placed.solve(w)                                                  # [..., parts] Point
    contact = lowest_point(placed, ground_normal(ground, centres / (w & centres)))
    # The ground's form at the contact over its gradient's length: the depth, to first order.
    return contact, ground_normal(ground, contact), -(ground(contact) & contact) / (2 * ground(contact).norm())


# Each top lifted so its tip clears the ground, then tilted.
starts = ((mv.zw * ((height * np.cos(tilt) + clearance) / 2)).exp() * (mv.zx * (tilt / 2)).exp()).normalized()   # [tops] Motor
_, _, start_depths = contacts(starts, parts, ground)                          # [tops, parts] Scalar
save_figure(image_figure(render.row(starts, parts, ground, view, view_centre, view_extent, n_pixels)), "spinning_top_start");

## 3. Spinning

Each step moves the tops freely, a Runge-Kutta step of the rate under the weight and the air's drag less the gyroscopic turn of the momentum, `inertia(rate).commutator(rate)`, then corrects the contacts one part after another, so that each sees the corrections before it, each along one force line at a time: pressed out along the ground's normal by its depth, held against sliding when the whole correction fits within the friction cone, and the rate the corrected step implies slowed by the dynamic friction, again part by part. Each correction is the force line's own compliance, `twist & body`, with the top's inverse inertia, so a part far from the centre of mass gives more than one close to it. The steps between two frames are compiled into one loop, all four tops at once.

From the tilted start each top leans a little further, then holds its lean while its axis swings around the vertical, precessing, while the drag slows its spin. All four stay up; the sharper tip, the heavier disc and the slipperier ground each change how far it leans and how fast it precesses.

In [ ]:
# Gravity; the static friction, and each top's dynamic friction, the last on slipperier ground; the air's drag rate;
# the spin; the run's length in seconds, the time step, the steps per frame, and a frame's duration.
gravity, static_friction, drag, spin_rate = 9.81, 0.4, 0.13, 18.0
dynamic_frictions = np.array([0.3, 0.3, 0.3, 0.1])
runtime, dt, n_steps_per_frame, duration_ms = 5.0, 4e-3, 10, 33

In [ ]:
def clamp(value: Scalar, limit: Scalar) -> Scalar:
    """The value with its magnitude held to the limit."""
    magnitude = value.abs()
    return value * ((magnitude + limit - (magnitude - limit).abs()) / 2) / (magnitude + 1e-12)


def along_ground(direction: Direction, normal: Direction) -> Direction:
    """The direction less its part along the ground's normal."""
    return direction - normal * (direction.dual() | normal.dual())


def compliance(motor: Motor, inverse_inertia: InverseInertia, lines: Forque) -> tuple[Rate, Scalar]:
    """The body's twist per unit load along each line, and its compliance along that line: the twist's pairing with
    the line, both in the body's frame."""
    body = motor << lines
    twist = inverse_inertia(body)
    return twist, twist & body


def moved(motor: Motor, twists: Rate, loads: Scalar) -> Motor:
    """The body carried by the twists of the loads along its contacts' lines, summed over the parts."""
    return (motor * ((twists * loads).sum(axis=-1) * -0.5).exp()).normalized()


def corrected(before: Motor, motor: Motor, inverse_inertia: InverseInertia, parts: Quadric, ground: Quadric,
              static: float, dynamic: np.ndarray, dt: float) -> tuple[Motor, Rate]:
    """The free step's pose corrected against the ground, one part after another so that each sees the corrections
    before it: pressed out along the ground's normal by its depth, then held against sliding within the friction
    cone; then the rate it implies, slowed by each part's dynamic friction in turn."""
    per_part = inverse_inertia[..., None]                                      # [..., 1] InverseInertia
    # The parts in the motor's context, so that a compiled loop can take them one at a time.
    parts = parts.materialize(motor.context)                                   # [parts] Quadric

    def press(motor: Motor, part: Quadric) -> tuple[Motor, Scalar]:
        # One part, as a set of one, so that the contact helpers apply as they are.
        contact, normal, depth = contacts(motor, part[None], ground)           # [..., 1]
        # Press out along the normal line through the contact, by its depth; a part in the air has none.
        twists, give = compliance(motor[..., None], per_part, contact & normal)
        pressed = depth.clip(0, np.inf) / give
        motor = moved(motor, twists, pressed)
        # Hold against sliding, back towards where the contact's material point was, in the ground's plane: only
        # when the whole correction fits within the static friction's share of the press.
        slid = along_ground(contact - (before[..., None] >> (motor[..., None] << contact)), normal)
        back = slid.dual().norm()
        twists, give = compliance(motor[..., None], per_part, contact & (-slid / (back + 1e-12)))
        sticking = back / (give + 1e-12)
        return moved(motor, twists, sticking * (sticking <= pressed * static)), pressed

    motor, pressures = jax.lax.scan(press, motor, parts)                     # [...] Motor, [parts, ..., 1] Scalar
    # The rate the corrected step implies.
    rate = (before.inverse() * motor).normalized().log() * (-2 / dt)           # [...] Rate

    def slow(rate: Rate, part_and_pressure: tuple[Quadric, Scalar]) -> tuple[Rate, None]:
        part, pressed = part_and_pressure
        contact, normal, _ = contacts(motor, part[None], ground)              # [..., 1]
        # The contact's remaining slip, held by at most the dynamic friction's share of its pressing impulse.
        sliding = along_ground(contact.commutator(motor[..., None] >> rate[..., None]), normal)
        speed = sliding.dual().norm()
        twists, give = compliance(motor[..., None], per_part, contact & (-sliding / (speed + 1e-12)))
        return rate + (twists * clamp(speed / (give + 1e-12), pressed * dynamic[..., None] / dt)).sum(axis=-1), None

    rate, _ = jax.lax.scan(slow, rate, (parts, pressures))
    return motor, rate

def free_step(motor: Motor, rate: Rate, inertia: Inertia, inverse_inertia: InverseInertia, loads, dt: float
              ) -> tuple[Motor, Rate]:
    """One Runge-Kutta step of the rate under the loads less the gyroscopic turn of the momentum, then the motor
    along the stepped rate."""
    def acceleration(rate: Rate) -> Rate:
        return inverse_inertia(loads(motor, rate) - inertia(rate).commutator(rate))

    k1 = acceleration(rate)
    k2 = acceleration(rate + k1 * (dt / 2))
    k3 = acceleration(rate + k2 * (dt / 2))
    k4 = acceleration(rate + k3 * dt)
    rate = rate + (k1 + 2 * k2 + 2 * k3 + k4) * (dt / 6)
    return motor * (rate * (-dt / 2)).exp(), rate

In [ ]:
def spin(motor: Motor, rate: Rate, parts: Quadric, inertia: Inertia, mass: np.ndarray, ground: Quadric,
         drag: float, dynamic: np.ndarray, dt: float, n_frames: int, n_steps_per_frame: int):
    """The tops' motors and rates at every frame `[tops]`, `n_steps_per_frame` steps apart."""
    inverse_inertia = inertia.inverse()                                        # [tops] InverseInertia

    def loads(motor: Motor, rate: Rate) -> Forque:
        # The weight, a force line down through the centre of mass, and the air's drag against the momentum; in the
        # body's frame.
        weight = motor << ((motor >> origin) & (up * (-gravity * mass)))
        return weight - inertia(rate) * drag

    def step(state: tuple[Motor, Rate], _: None) -> tuple[tuple[Motor, Rate], None]:
        before, rate = state
        motor, rate = free_step(before, rate, inertia, inverse_inertia, loads, dt)
        return corrected(before, motor, inverse_inertia, parts, ground, static_friction, dynamic, dt), None

    # A frame's steps compiled into one loop, carrying the motors and rates as every step returns them.
    frame = jax.jit(lambda state: jax.lax.scan(step, state, length=n_steps_per_frame)[0])
    context = JaxContext(ga)
    state = (motor.retyped(Motor).materialize(context), rate.retyped(Rate).materialize(context))
    for _ in range(n_frames):
        yield state[0].materialize(mv.context), state[1].materialize(mv.context)
        state = frame(state)


n_frames = round(runtime / dt) // n_steps_per_frame
states = list(spin(starts, (mv.xy * spin_rate).broadcast_to(starts.shape), parts, inertia, part_masses.sum(axis=-1),
                   ground, drag, dynamic_frictions, dt, n_frames, n_steps_per_frame))   # [frames] ([tops] Motor, [tops] Rate)
show_animation([render.row(motors, parts, ground, view, view_centre, view_extent, n_pixels) for motors, _ in states],
               "spinning_top", duration_ms)

In [ ]:
# checks
# Every top starts clear of the ground, stays up over the run, leaning less than halfway over, and keeps spinning the
# same way while the drag slows it.
leans = np.stack([((motors >> up).dual() | up.dual()).abs().arccos().to_array() for motors, _ in states])   # [frames, tops]
spins = np.stack([-(mv.xy | rates).to_array() for _, rates in states])         # [frames, tops]
assert (start_depths.to_array() <= 0).all()
assert leans.max() < np.pi / 4
assert (spins > 0).all() and (spins[-1] < spins[0]).all()